In [1]:
!git clone https://github.com/jburroni/IntroPPLs26.git

%cd IntroPPLs26/notebooks/Jun-19

Cloning into 'IntroPPLs26'...
remote: Enumerating objects: 120, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 120 (delta 21), reused 53 (delta 13), pack-reused 58 (from 1)
Receiving objects: 100% (120/120), 39.95 MiB | 18.57 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/content/IntroPPLs26/notebooks/Jun-19


# Activity 2 - Measure the collapse, then fix the prior

You completed `evaluate` in Activity 1. Likelihood weighting proposes latent values from
the **prior** and weights them by the observation. When the prior sits far from the data
the weights become very uneven. You will measure that with your own diagnostic, then fix
it by choosing a better prior.

## Your job

1. Implement the **effective sample size** and report it for the mismatched model.
2. Keep the same observation, but design a **new prior** that makes the ESS much larger.

In [ ]:
import os, sys

# Locate the repo's interpreters/ folder by walking up from the working directory,
# so this notebook runs from wherever you open it.
_p = os.path.abspath(os.getcwd())
while _p != os.path.dirname(_p):
    if os.path.isdir(os.path.join(_p, "interpreters", "minippl")):
        sys.path.insert(0, os.path.join(_p, "interpreters"))
        break
    _p = os.path.dirname(_p)

import numpy as np
import matplotlib.pyplot as plt
from minippl import parse_one, PRIMITIVES, Symbol

## Your evaluator from Activity 1

Paste the `if` line you wrote in Activity 1 where marked below. Everything that follows
calls `run`, which wraps `evaluate`.

In [ ]:
class State:
    """The inference state: a random-number generator and the running log-weight."""
    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)
        self.log_w = 0.0


def evaluate(expr, env, state):
    if isinstance(expr, Symbol):                 # variable: look it up
        return env[expr]
    if not isinstance(expr, list):               # constant: number, bool, string, nil
        return expr
    op, *args = expr
    if op == "let":                              # (let [v e ...] body)
        binds, *body = args
        env = dict(env)
        for i in range(0, len(binds), 2):
            env[binds[i]] = evaluate(binds[i + 1], env, state)
        return [evaluate(e, env, state) for e in body][-1]
    if op == "if":                               # (if test then else)
        test, then, els = args
        cond_res = evaluate(test, env, state)
        if cond_res:
            return evaluate(then, env, state)
        return evaluate(els, env, state)
    if op == "sample":                           # (sample d): draw from the prior, move on
        return evaluate(args[0], env, state).sample(state.rng)
    if op == "observe":                          # (observe d v): score the datum, keep its value
        dist  = evaluate(args[0], env, state)
        value = evaluate(args[1], env, state)
        state.log_w += dist.log_prob(value)
        return value
    fn = PRIMITIVES[op] if op in PRIMITIVES else evaluate(op, env, state)
    return fn(*[evaluate(a, env, state) for a in args])


def run(src, seed=0):
    """One forward execution. Returns (value, log_weight)."""
    st = State(seed)
    return evaluate(parse_one(src), {}, st), st.log_w

## Tool

`sample_weights(program, L)` runs a model `L` times and returns the sampled `mu` values
with their **normalized** weights (it does the prior sampling and the log-weight
normalization for you).

In [ ]:
def sample_weights(program, L=5000, seed0=0):
    xs, lws = zip(*(run(program, seed=s) for s in range(seed0, seed0 + L)))
    w = np.exp(np.array(lws) - np.max(lws))   # shift by the max, then normalize
    w /= w.sum()
    return np.array(xs), w

## Step 1 - implement the effective sample size

For normalized weights $\bar w_1, \dots, \bar w_L$ (summing to 1), the effective sample
size is $\text{ESS} = 1 / \sum_\ell \bar w_\ell^{\,2}$. It ranges from `1` (one run holds
all the weight) to `L` (every run counts equally). Implement it.

In [ ]:
def ess(w):
    # w: normalized weights (a numpy array summing to 1).
    # Effective sample size = 1 / sum of squared weights.
    raise NotImplementedError("implement ESS")

In [ ]:
prior_far = "(let [mu (sample (normal 0 1))] (observe (normal mu 1) 8.0) mu)"
xs, w = sample_weights(prior_far)
print("ESS, prior N(0,1):", round(ess(w), 1), "out of", len(w))

## Step 2 - design a prior that raises the ESS

Keep the **same observation** `(observe (normal mu 1) 8.0)`, but change the prior on
`mu` so that prior draws land near where the data pull. Edit the program below, rerun,
and compare the ESS.

In [ ]:
prior_near = "(let [mu (sample (normal 0 1))] (observe (normal mu 1) 8.0) mu)"  # change the prior on mu
xs2, w2 = sample_weights(prior_near)
print("ESS, your prior:", round(ess(w2), 1), "out of", len(w2))

## See it

The histogram of normalized weights, before and after. A tall spread of even weights
means a high ESS; a single spike means collapse.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, (label, ww) in zip(axes, [("prior N(0,1)", w), ("your prior", w2)]):
    ax.hist(ww, bins=60)
    ax.set_title(f"{label}    ESS = {ess(ww):.0f} / {len(ww)}")
    ax.set_xlabel("normalized weight")
    ax.set_ylabel("number of runs")
fig.tight_layout()
plt.show()

## What you found

Your ESS started at a handful: almost all the weight on a few runs. Moving the prior onto
the data, with the **same observation**, lets most runs explain it, the weights even out,
and the ESS climbs toward `L`. In likelihood weighting the prior **is** the proposal, so
the ESS measures how well it matches where the data pull.

But look at what you changed: a different prior is a different model, hence a different
posterior. A prior squeezed tightly onto `8` would drive the ESS to `L` and answer the
question by assuming it. The real goal, next class: propose near the posterior **without**
changing the model.